# XGBoost - eXtreme Gradient Boosting
- XGBoost, GBM'in h?z ve tahmin performans?n? artt?rmak ?zere optimize edilmi?; ?l?eklenebilir ve farkl? platformlara entegre edilebilir halidir.
- R, Python, Hadoop, Scala, Julia ile kullan?labilir.
- ?le?klenebilirdir.
- H?zl?d?r.
- Tahmin ba?ar?s? y?ksektir.
- Bir?ok Kaggle yar??mas?nda ba?ar?s? kan?tlanm??t?r.

In [ ]:
!pip install xgboost

**Gerekli K?t?phaneler** 

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score, mean_squared_error, r2_score, roc_auc_score, roc_curve, classification_report
from sklearn.preprocessing import scale, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [ ]:
# Uyar? Mesajlar? ile Kar??la?mamak i?in bu k?t?phaneyi kullan?yoruz.
from warnings import filterwarnings
filterwarnings("ignore", category=DeprecationWarning)
filterwarnings("ignore", category=FutureWarning)

# Veri Seti Hikayesi ve Problemi: ?eker Hastal??? Tahmini

In [ ]:
df = pd.read_csv("./diabetes.csv")
# bu csv dosyas?n?n i?erisinde eksik g?zlemleri(NA) ??kard?k.
df.head()

# XGBoost ve Tahmin ??lemleri

In [ ]:
y= df["Outcome"]
X = df.drop(["Outcome"], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size = 0.30,
                                                    random_state = 42)

**Model**

In [ ]:
xgb_model = XGBClassifier().fit(X_train, y_train)

In [ ]:
?xgb_model

**Tahmin**

In [ ]:
y_pred = xgb_model.predict(X_test)

In [ ]:
accuracy_score(y_test,y_pred)

# Model Tuning (Model Do?rulama)
-  Modelin ba?ar?s?n? de?erlendirmek i?in izlenecek yol Do?ruluk oran?na (accuracy_score) bak?larak yap?lacak. 
- **accuracy_score**' u ne kadar b?y?k olursa s?n?fland?rma da o kadar ba?ar?l? oldu?u anlam?na gelir.
- Hiper parametrelerin optimum de?erlerine ula?mak i?inde Cross-Validation y?ntemini kullanm?? olaca??z.
- Yani modelleri Train Setiyle e?itirken Cross-Validation y?netemini kullan?p, d??ar? b?rakt???m?z Test Setini ise performans?m?z? test etmek i?in kullanacaz.

In [ ]:
xgb_params = {"n_estimators": [100,500,1000],
              "subsample": [0.6, 0.8, 2],
              "max_depth": [3,5,7],
              "learning_rate": [0.1, 0.001,0.01]}

In [ ]:
xgb_cv_model = GridSearchCV(xgb_model,xgb_params, cv=10,
                            n_jobs= -1, verbose=2).fit(X_train, y_train)

In [ ]:
xgb_cv_model.best_params_

**Final Modeli**

In [ ]:
xgb_tuned = XGBClassifier( learning_rate= 0.001,
                           max_depth= 7,
                           n_estimators= 500, 
                           subsample= 0.6).fit(X_train, y_train)

In [ ]:
y_tuned_pred = xgb_tuned.predict(X_test)
accuracy_score(y_test,y_tuned_pred)

## De?i?ken ?nem D?zeyleri

In [ ]:
feature_imp = pd.Series(xgb_tuned.feature_importances_,
                        index=X_train.columns).sort_values(ascending=False)

sns.barplot(x=feature_imp, y=feature_imp.index)
plt.xlabel("De?i?ken ?nem Skorlar?")
plt.ylabel("De?i?kenler")
plt.title("De?i?ken ?nem D?zeyleri")
plt.savefig("De?i?ken ?nem D?zeyleri")
plt.show()